# Cleaning the Tunnel Runner data

Data: Markovitch et al. (2025), shared on OSF: https://osf.io/9564p/



In [1]:
# 1. Load the two files
import pandas as pd

games = pd.read_csv("data_for_ML.csv", index_col=0)
preds = pd.read_csv("individual_predictions.csv", index_col=0)

print(games.shape, preds.shape)
games.head()

(871, 23) (871, 6)


,Arew,Apun,betaF,betaP,frequency_pref,good_pref,baseline_RT,baseline_sigma,baseline_beta,flanker_rt,...,flanker_acc,ice_acc,SSRT_match,SSRT_mismatch,tts,tts_sigma,gamer_identity,gender,y,.folds
1,0.113378,0.090309,3.613057,3.917357,3.276587,3.616471,0.264660,0.483365,0.109131,-0.003991,...,0.060081,0.251335,0.479229,0.477738,0.127406,0.135347,8,O,34,10
2,0.476403,0.044988,2.545307,-1.913020,-0.688288,-2.002537,-0.093188,0.041248,-0.039706,0.010206,...,0.129909,-0.324042,0.377686,0.337524,0.046512,-0.399028,7,F,36,6
3,0.023662,0.024254,2.547176,3.716035,2.766180,1.720519,-0.036263,-0.275398,0.175428,0.015226,...,0.204224,0.122558,0.320386,0.362767,0.013964,-0.393043,13,M,23,1
4,0.145819,0.053265,2.235504,-1.187294,-0.979313,-0.700642,0.344769,0.557292,-0.261593,-0.039039,...,0.244510,1.520750,0.375571,0.325524,-0.002830,0.205029,1,F,44,8
5,0.423334,0.081414,1.598718,-0.694047,-0.754590,0.302259,0.096542,0.146431,-0.138452,-0.005854,...,0.005167,-0.004011,0.308690,0.329243,0.001598,-0.260944,12,M,42,4


In [2]:
# 2. Give the columns names we can read
# (meanings come from the paper's description of the measures, not from a codebook)
new_names = {
    "baseline_RT": "baseline_reaction_time",       # how fast the player responds on easy trials
    "baseline_sigma": "baseline_rt_variability",   # how much the reaction time fluctuates
    "baseline_beta": "baseline_rt_slow_tail",      # how often very slow responses happen
    "baseline_acc": "baseline_accuracy",           # share of correct responses on easy trials
    "y": "age",
}
games = games.rename(columns=new_names)

In [3]:
# 3. Check the data
print("duplicate rows:", games.duplicated().sum(), preds.duplicated().sum())

# missing values per column (only show columns that have some)
missing = games.isna().sum()
print(missing[missing > 0])

# do the values make sense?
print(games["age"].min(), games["age"].max())

duplicate rows: 0 0
SSRT_match       12
SSRT_mismatch    11
tts              20
tts_sigma        20
dtype: int64
18 75


In [4]:
# 4. Both files have no ID, so check that row i is the same person in both
print("same age on every row:", (games["age"].values == preds["age"].values).all())
print("same gamer identity on every row:", (games["gamer_identity"].values == preds["gamer_identity"].values).all())

same age on every row: True
same gamer identity on every row: True


In [5]:
# 5. Put the gender label from the predictions file in (it is more detailed)
games["gender"] = preds["gender"].replace({
    "I prefer to self-describe my gender": "Self-described",
    "I prefer not to disclose": "Not disclosed",
})
games["gender"].value_counts()

gender
Female            425
Male              419
Non-binary         19
Self-described      4
Not disclosed       4
Name: count, dtype: int64

In [6]:
# 6. Keep only what a Pixie profile could hold: age, gender and four simple play measures
keep = ["age", "gender", "baseline_reaction_time", "baseline_rt_variability",
        "baseline_rt_slow_tail", "baseline_accuracy"]
final = games[keep]

print("missing values left:", final.isna().sum().sum())
final.to_csv("tunnel_runner_pixie_relevant.csv", index_label="participant_id")

missing values left: 0


In [7]:
# 7. Summary of the numbers
final.describe().round(2)

,age,baseline_reaction_time,baseline_rt_variability,baseline_rt_slow_tail,baseline_accuracy
count,871.00,871.00,871.00,871.00,871.00
mean,36.64,0.00,-0.00,-0.00,0.00
std,11.22,0.12,0.37,0.30,0.53
min,18.00,-0.32,-0.87,-0.76,-1.11
25%,28.00,-0.09,-0.28,-0.23,-0.41
50%,35.00,-0.01,-0.05,0.00,0.03
75%,44.00,0.08,0.24,0.21,0.39
max,75.00,0.57,1.14,0.71,1.41


In [8]:
# 8. How many people per gender
final["gender"].value_counts().to_frame()

,count
gender,
Female,425
Male,419
Non-binary,19
Self-described,4
Not disclosed,4
